## 准备数据

In [1]:
import os
import numpy as np
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, optimizers, datasets

os.environ['TF_CPP_MIN_LOG_LEVEL'] = '2'  # or any {'0', '1', '2'}

def mnist_dataset():
    (x, y), (x_test, y_test) = datasets.mnist.load_data()
    #normalize
    x = x/255.0
    x_test = x_test/255.0
    
    return (x, y), (x_test, y_test)

In [2]:
print(list(zip([1, 2, 3, 4], ['a', 'b', 'c', 'd'])))

[(1, 'a'), (2, 'b'), (3, 'c'), (4, 'd')]


## 建立模型

In [3]:
class myModel:
    def __init__(self):
        ####################
        '''声明模型对应的参数'''
        self.W1 = tf.Variable(tf.random.truncated_normal([784, 128], stddev=0.1))
        self.b1 = tf.Variable(tf.zeros([128]))
        
        self.W2 = tf.Variable(tf.random.truncated_normal([128, 10], stddev=0.1))
        self.b2 = tf.Variable(tf.zeros([10]))
        ####################
    def __call__(self, x):
        ####################
        '''实现模型函数体，返回未归一化的logits'''
        # 展平输入图像
        x = tf.reshape(x, [-1, 784])
        
        # 隐藏层
        h1 = tf.nn.relu(tf.matmul(x, self.W1) + self.b1)
        
        # 输出层 
        logits = tf.matmul(h1, self.W2) + self.b2
        ####################
        return logits
        
model = myModel()

optimizer = optimizers.Adam()

## 计算 loss

In [4]:
@tf.function
def compute_loss(logits, labels):
    return tf.reduce_mean(
        tf.nn.sparse_softmax_cross_entropy_with_logits(
            logits=logits, labels=labels))

@tf.function
def compute_accuracy(logits, labels):
    predictions = tf.argmax(logits, axis=1)
    return tf.reduce_mean(tf.cast(tf.equal(predictions, labels), tf.float32))

@tf.function
def train_one_step(model, optimizer, x, y):
    with tf.GradientTape() as tape:
        logits = model(x)
        loss = compute_loss(logits, y)

    # compute gradient
    trainable_vars = [model.W1, model.W2, model.b1, model.b2]
    grads = tape.gradient(loss, trainable_vars)
    for g, v in zip(grads, trainable_vars):
        v.assign_sub(0.01*g)

    accuracy = compute_accuracy(logits, y)

    # loss and accuracy is scalar tensor
    return loss, accuracy

@tf.function
def test(model, x, y):
    logits = model(x)
    loss = compute_loss(logits, y)
    accuracy = compute_accuracy(logits, y)
    return loss, accuracy

## 实际训练

In [5]:
train_data, test_data = mnist_dataset()
for epoch in range(50):
    loss, accuracy = train_one_step(model, optimizer, 
                                    tf.constant(train_data[0], dtype=tf.float32), 
                                    tf.constant(train_data[1], dtype=tf.int64))
    print('epoch', epoch, ': loss', loss.numpy(), '; accuracy', accuracy.numpy())
loss, accuracy = test(model, 
                      tf.constant(test_data[0], dtype=tf.float32), 
                      tf.constant(test_data[1], dtype=tf.int64))

print('test loss', loss.numpy(), '; accuracy', accuracy.numpy())

epoch 0 : loss 2.4884782 ; accuracy 0.07225
epoch 1 : loss 2.4682693 ; accuracy 0.07723334
epoch 2 : loss 2.449459 ; accuracy 0.0824
epoch 3 : loss 2.431849 ; accuracy 0.08738333
epoch 4 : loss 2.4152768 ; accuracy 0.09425
epoch 5 : loss 2.3996124 ; accuracy 0.10015
epoch 6 : loss 2.3847468 ; accuracy 0.10621667
epoch 7 : loss 2.3705864 ; accuracy 0.112833336
epoch 8 : loss 2.3570542 ; accuracy 0.119166665
epoch 9 : loss 2.3440835 ; accuracy 0.12485
epoch 10 : loss 2.3316202 ; accuracy 0.1312
epoch 11 : loss 2.3196142 ; accuracy 0.13695
epoch 12 : loss 2.3080204 ; accuracy 0.14278333
epoch 13 : loss 2.296799 ; accuracy 0.1486
epoch 14 : loss 2.2859187 ; accuracy 0.15436667
epoch 15 : loss 2.2753499 ; accuracy 0.16006666
epoch 16 : loss 2.265067 ; accuracy 0.16608334
epoch 17 : loss 2.2550473 ; accuracy 0.17146666
epoch 18 : loss 2.2452688 ; accuracy 0.17703333
epoch 19 : loss 2.235712 ; accuracy 0.18235
epoch 20 : loss 2.2263603 ; accuracy 0.18796666
epoch 21 : loss 2.2171984 ; accurac